# ☀️ Daylight studio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snjsomnath/foton/blob/main/notebooks/daylight_studio.ipynb)

Shape a building, place its windows and a neighbour, and see the daylight factor and annual daylight
autonomy on every floor in about a second. Foton traces the scene on the runtime's GPU; there is no
server and nothing to install on your own machine.

1. **Runtime → Change runtime type → T4 GPU** (the free tier is enough).
2. **Runtime → Run all.** Setup takes about a minute.
3. Change any field in the studio form. The study re-runs and redraws.

The model is a rectangular Honeybee building with one room and one 0.5 m sensor grid per floor,
sensors 0.75 m above the floor. Windows are 1.5 m high strips at a 0.9 m sill, sized to each façade's
window-to-wall ratio. Annual results use the selected EPW, Honeybee's default 08:00–18:00 occupancy
and a 50 % time target for sDA.

In [ ]:
# @title ⚙️ Setup: installs Foton and connects it to the GPU { display-mode: "form" }
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    def sh(cmd):
        subprocess.run(cmd, shell=True, check=True)
    gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
    print('GPU:', gpu or 'none. Use Runtime -> Change runtime type -> T4 GPU, then run again.')
    # Colab has the NVIDIA driver but no Vulkan ICD manifest; point the loader at NVIDIA's headless EGL library.
    sh('apt-get -qq update && apt-get -qq install -y libvulkan1 > /dev/null')
    os.makedirs('/etc/vulkan/icd.d', exist_ok=True)
    with open('/etc/vulkan/icd.d/nvidia_icd.json', 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "/usr/lib64-nvidia/libEGL_nvidia.so.0", "api_version": "1.3.0"}}')
    sh('pip install -q "foton-daylight[honeybee]>=0.3.5"')
    EPW_DIR = '/content/epw'
    os.makedirs(EPW_DIR, exist_ok=True)
    for city in ['gothenburg', 'seattle', 'phoenix']:
        if not os.path.exists(f'{EPW_DIR}/{city}.epw'):
            sh(f'curl -sfL -o {EPW_DIR}/{city}.epw https://raw.githubusercontent.com/snjsomnath/foton/main/epw/{city}.epw')
else:
    EPW_DIR = os.path.abspath(os.path.join('..', 'epw'))

import math, time
import numpy as np
import foton
from foton.honeybee import HoneybeeStudy
from ladybug_geometry.geometry3d import Point3D
from honeybee.model import Model
from honeybee.room import Room
from honeybee.shade import Shade
from honeybee.facetype import Wall

# One engine for the whole session: it keeps the GPU driver loaded between studies.
ENGINE = foton.Engine({'backend': 'auto'})
DEVICE = ENGINE.capabilities()['name'].split('(')[0].strip()
WEATHER_MEMO = {}
GRID = 0.5   # sensor spacing (m); 0.3 is finer but about 3x the sensors

def cardinal(normal):
    # Compass bearing of an outward wall normal (+Y is north), as N/E/S/W.
    bearing = math.degrees(math.atan2(normal.x, normal.y)) % 360
    return ['north', 'east', 'south', 'west'][int(((bearing + 45) % 360) // 90)]

def build_model(width, depth, floors, storey, rotation, wwr, overhang, fins, neighbour):
    # A rectangular building, one Room and one sensor grid per floor, plus an optional neighbour block.
    rooms, grids = [], []
    for f in range(floors):
        room = Room.from_box(f'Floor_{f + 1}', width, depth, storey, origin=Point3D(-width / 2, -depth / 2, f * storey))
        room.rotate_xy(-rotation, Point3D(0, 0, 0))   # positive rotation turns the building clockwise
        for face in room.faces:
            if isinstance(face.type, Wall) and wwr[cardinal(face.normal)] > 0:
                face.apertures_by_ratio_rectangle(wwr[cardinal(face.normal)], 1.5, 0.9, 2.5)
                for ap in face.apertures:
                    if overhang > 0:
                        ap.overhang(overhang)
                    if fins > 0:
                        ap.left_fin(fins); ap.right_fin(fins)
        rooms.append(room)
        grids.append(room.properties.radiance.generate_sensor_grid(GRID, offset=0.75))
    model = Model('daylight_studio', rooms=rooms)
    model.properties.radiance.sensor_grids = grids
    if neighbour:
        distance, n_floors = neighbour
        half, y0 = max(width, depth) / 2 + 4, -max(width, depth) / 2 - distance
        block = Room.from_box('neighbour', 2 * half, 8, 3.2 * n_floors, origin=Point3D(-half, y0 - 8, 0))
        model.add_shades([Shade(f'neighbour_{i}', face.geometry) for i, face in enumerate(block.faces)])
    return model

# The last building's study. While its geometry is unchanged, the GPU scene, the daylight factor and the
# annual daylight coefficients are reused, so changing the climate, the lux threshold or the colouring
# only re-runs the annual sums.
STUDY = {'key': None, 'study': None, 'df': {}}

def simulate(model, epw, quality='final', annual=True, lux=300, key=None):
    # key names the geometry (any hashable value); None always prepares a new study.
    t0 = time.perf_counter()
    if key is None or key != STUDY['key']:
        STUDY.update(key=None, study=None, df={})   # release the old GPU scene before building the next
        STUDY.update(key=key, study=HoneybeeStudy(model, backend='auto', weather_memo=WEATHER_MEMO))
    study = STUDY['study']
    positions = np.asarray(study.prepared.arrays['sensor_positions'], dtype=np.float32)
    run = STUDY['df'].get(quality)
    if run is None:
        run = STUDY['df'][quality] = study.daylight_factor(quality=quality)
    grids, start = [], 0
    for g in run.grids:
        grids.append({'name': g.room_identifier, 'id': g.full_identifier, 'start': start, 'count': g.sensor_count})
        start += g.sensor_count
    order = np.concatenate([g.sensor_indices for g in run.grids])
    df = np.concatenate([np.asarray(g.values, dtype=np.float32) for g in run.grids])
    da = sda = None
    if annual:
        a = study.annual_daylight(epw, quality=quality, threshold=float(lux), target_time=50.0)
        da = np.concatenate([np.asarray(a.grid(g['id']).da, dtype=np.float32) for g in grids])
        sda = 100.0 * float((da >= 50).mean())
    return {'points': positions[order], 'df': df, 'da': da, 'sda': sda, 'grids': grids,
            'seconds': time.perf_counter() - t0}

print('Foton', foton.__version__ if hasattr(foton, '__version__') else '', 'on', ENGINE.capabilities()['name'])
if not ENGINE.capabilities()['hardware_acceleration']:
    print('Foton is on the CPU: each run will take minutes. Check the runtime type and the Vulkan setup above.')


In [ ]:
# @title ☀️ Daylight studio { run: "auto", display-mode: "form" }
# Change any field and Colab re-runs this cell: the building is rebuilt, Foton traces it on the GPU,
# and the results below are redrawn. Plain Colab forms and Plotly, so it also works on a phone.

# @markdown ### 🏢 Building
width = 10  # @param {type:"slider", min:3, max:24, step:0.5}
depth = 12  # @param {type:"slider", min:3, max:18, step:0.5}
floors = 3  # @param {type:"slider", min:1, max:6, step:1}
floor_height = 3.2  # @param {type:"slider", min:2.6, max:4.5, step:0.1}
rotation = 0  # @param {type:"slider", min:-180, max:180, step:5}
# @markdown ### 🪟 Windows (window-to-wall ratio) and shading (m)
wwr_north = 0.25  # @param {type:"slider", min:0, max:0.9, step:0.05}
wwr_east = 0.15  # @param {type:"slider", min:0, max:0.9, step:0.05}
wwr_south = 0.35  # @param {type:"slider", min:0, max:0.9, step:0.05}
wwr_west = 0.15  # @param {type:"slider", min:0, max:0.9, step:0.05}
overhang = 0  # @param {type:"slider", min:0, max:2, step:0.1}
side_fins = 0  # @param {type:"slider", min:0, max:1.5, step:0.1}
# @markdown ### 🏙️ Neighbour to the south
neighbour = True  # @param {type:"boolean"}
neighbour_distance = 8  # @param {type:"slider", min:2, max:30, step:0.5}
neighbour_floors = 5  # @param {type:"slider", min:1, max:12, step:1}
# @markdown ### ☀️ Analysis
climate = "gothenburg"  # @param ["gothenburg", "seattle", "phoenix"]
quality = "final"  # @param ["final", "preview"]
annual = True  # @param {type:"boolean"}
da_threshold_lux = 300  # @param {type:"slider", min:100, max:1000, step:50}
colour_by = "daylight factor"  # @param ["daylight factor", "daylight autonomy"]

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

SCALE = [[0, '#0b1026'], [0.2, '#1d3a8a'], [0.45, '#0e9f9a'], [0.7, '#f5d64a'], [1, '#fff7d6']]
CSS = '''<style>
.ds{font-family:Inter,system-ui,sans-serif}
.ds-head{background:linear-gradient(115deg,#0b1026 0%,#1d3a8a 55%,#f59e0b 120%);color:#fff;border-radius:16px;padding:16px 20px;margin-bottom:10px}
.ds-head h1{margin:0;font-size:24px;font-weight:700;color:#fff}.ds-head p{margin:4px 0 0;opacity:.85;font-size:13px}
.ds-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(130px,1fr));gap:8px}
.ds-card{background:#0b1026;color:#cbd5e1;border-radius:12px;padding:10px 14px;font-size:12px}
.ds-card b{display:block;font-size:22px;color:#fbbf24}
</style>'''


def _mesh(polys, color, opacity):
    # Triangle-fan mesh of planar polygons (Honeybee faces are convex here).
    xs, ys, zs, I, J, K = [], [], [], [], [], []
    for verts in polys:
        n0 = len(xs)
        for p in verts:
            xs.append(p.x); ys.append(p.y); zs.append(p.z)
        for t in range(1, len(verts) - 1):
            I.append(n0); J.append(n0 + t); K.append(n0 + t + 1)
    return go.Mesh3d(x=xs, y=ys, z=zs, i=I, j=J, k=K, color=color, opacity=opacity, flatshading=True, hoverinfo='skip')


def view_3d(model, pts, values, label):
    # The building (walls, glass, shading), the neighbour, and every sensor coloured by its result.
    walls, glass, shades = [], [], []
    for room in model.rooms:
        for face in room.faces:
            walls.append(face.geometry.vertices)
            for ap in face.apertures:
                glass.append(ap.geometry.vertices)
                shades += [s.geometry.vertices for s in ap.outdoor_shades]
    context = [s.geometry.vertices for s in model.orphaned_shades]
    span = max(width, depth, 10)
    fig = go.Figure([
        _mesh(walls, '#e2e8f0', 0.12), _mesh(glass, '#38bdf8', 0.55),
        _mesh(shades, '#475569', 0.8), _mesh(context, '#94a3b8', 0.12),
        go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
                     marker=dict(size=3, color=values, colorscale=SCALE, cmin=0, cmax=max(float(values.max()), 1e-6),
                                 colorbar=dict(title=label, thickness=12, len=0.6)),
                     hovertemplate=label + ' %{marker.color:.1f}<extra></extra>'),
        go.Scatter3d(x=[0, 0], y=[span * 0.75, span * 1.05], z=[0, 0], mode='lines+text', text=['', 'N'],
                     line=dict(color='#f59e0b', width=6), textfont=dict(color='#f59e0b', size=16), hoverinfo='skip'),
    ])
    fig.update_layout(height=480, margin=dict(l=0, r=0, t=0, b=0), showlegend=False, paper_bgcolor='rgba(0,0,0,0)',
                      scene=dict(aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                                 camera=dict(eye=dict(x=-1.0, y=1.2, z=1.0))))
    return fig


def floor_plans(result, values, label):
    # One plan per floor, two per row, north up, on a shared colour scale.
    grids, pts = result['grids'], result['points']
    cols = min(2, len(grids))
    rows = -(-len(grids) // cols)
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[g['name'].replace('_', ' ') for g in grids],
                        horizontal_spacing=0.03, vertical_spacing=0.08)
    vmax = max(float(values.max()), 1e-6)
    for n, g in enumerate(grids):
        r, c = n // cols + 1, n % cols + 1
        s = slice(g['start'], g['start'] + g['count'])
        fig.add_trace(go.Scattergl(x=pts[s, 0], y=pts[s, 1], mode='markers', showlegend=False,
                                   marker=dict(symbol='square', size=7, color=values[s], colorscale=SCALE, cmin=0, cmax=vmax,
                                               showscale=n == 0, colorbar=dict(title=label, thickness=10, orientation='h', y=-0.12 / rows, len=0.9)),
                                   hovertemplate=label + ' %{marker.color:.1f}<extra></extra>'), row=r, col=c)
        fig.update_xaxes(visible=False, row=r, col=c)
        fig.update_yaxes(visible=False, scaleanchor=f'x{n + 1 if n else ""}', row=r, col=c)
    fig.update_layout(height=260 * rows + 60, margin=dict(l=0, r=0, t=30, b=50),
                      paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
    return fig


geometry = (width, depth, floors, floor_height, rotation,
            (('north', wwr_north), ('east', wwr_east), ('south', wwr_south), ('west', wwr_west)),
            overhang, side_fins, (neighbour_distance, neighbour_floors) if neighbour else None)
model = build_model(*geometry[:5], dict(geometry[5]), *geometry[6:])
result = simulate(model, f'{EPW_DIR}/{climate}.epw', quality, annual, da_threshold_lux, key=geometry)
df, sda = result['df'], result['sda']
use_da = colour_by == 'daylight autonomy' and result['da'] is not None
values, label = (result['da'], 'DA %') if use_da else (df, 'DF %')

HISTORY = globals().get('HISTORY', [])   # kept across re-runs of this cell
HISTORY.append({'climate': climate, 'size': f'{width}×{depth}×{floors}', 'rotation': rotation,
                'wwr N/E/S/W': f'{wwr_north}/{wwr_east}/{wwr_south}/{wwr_west}',
                'neighbour': neighbour_floors if neighbour else 0, 'mean DF %': round(float(df.mean()), 2), 'area ≥2% DF': f'{(df >= 2).mean():.0%}',
                'sDA %': None if sda is None else round(sda, 1)})

cards = [('mean daylight factor', f'{df.mean():.1f} %'), ('floor area ≥ 2 % DF', f'{(df >= 2).mean():.0%}'),
         (f'sDA {da_threshold_lux} lx / 50 %', '—' if sda is None else f'{sda:.0f} %'),
         ('sensors', f'{df.size:,}'), ('Foton time', f"{result['seconds']:.2f} s")]
display(HTML(CSS + f'''<div class="ds"><div class="ds-head"><h1>☀️ Daylight studio</h1>
<p>⚡ Foton on {DEVICE} · change a field above and this redraws</p></div>
<div class="ds-cards">{"".join(f'<div class="ds-card"><b>{v}</b>{k}</div>' for k, v in cards)}</div></div>'''))
view_3d(model, result['points'], values, label).show()
floor_plans(result, values, label).show()
display(pd.DataFrame(HISTORY[::-1]))


## Diagnostics
Optional. Checks that the numbers behave (no windows means no daylight, a close neighbour lowers it) and measures how many studies a minute this GPU runs.

In [ ]:
# Runs without the studio: checks that results are sane and measures throughput on this GPU.
checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok), detail))

base = dict(width=8, depth=6, floors=2, storey=3.2, rotation=0, wwr={'north': 0.4, 'east': 0.3, 'south': 0.3, 'west': 0.3},
            overhang=0, fins=0, neighbour=None)
r = simulate(build_model(**base), f'{EPW_DIR}/gothenburg.epw', 'final', True)
check('daylight factor in range', 0 < r['df'].min() <= r['df'].max() < 100, f"mean {r['df'].mean():.2f} %, {r['df'].size} sensors")
check('one grid per floor', [g['name'] for g in r['grids']] == ['Floor_1', 'Floor_2'])
check('annual DA in range', r['da'] is not None and 0 <= r['da'].min() <= r['da'].max() <= 100, f"sDA {r['sda']:.0f} %")
blank = simulate(build_model(**{**base, 'wwr': {k: 0.0 for k in base['wwr']}}), f'{EPW_DIR}/gothenburg.epw', 'preview', False)
check('no windows gives no daylight', blank['df'].max() < 0.01, f"max DF {blank['df'].max():.3f} %")
shaded = simulate(build_model(**{**base, 'neighbour': (3, 10)}), f'{EPW_DIR}/gothenburg.epw', 'final', False)
check('a close neighbour lowers daylight', shaded['df'].mean() < r['df'].mean(), f"{r['df'].mean():.2f} -> {shaded['df'].mean():.2f} %")

rng = np.random.default_rng(0)
times = []
for _ in range(20):
    m = build_model(float(rng.uniform(5, 14)), float(rng.uniform(4, 12)), int(rng.integers(1, 4)), 3.2, 0,
                    {k: float(rng.uniform(0.2, 0.6)) for k in base['wwr']}, 0, 0, None)
    times.append(simulate(m, f'{EPW_DIR}/gothenburg.epw', 'preview', False)['seconds'])
check('throughput, 20 buildings', True, f'{60 / np.mean(times):.0f} studies/min, median {np.median(times):.2f} s')

print(DEVICE, '\n')
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL':4}  {name:34} {detail}")
